# SatFlood-AI: Multimodal Satellite Flood Damage Assessment
### Multimodal AI Hackathon 2026 — Track B: Mapping Flood Damage from Space

This notebook demonstrates the end-to-end prototype for:
1. **Sentinel-1 SAR change detection** (same-orbit-track pair)
2. **Himalayan false positive suppression** (Copernicus DEM slope, wet-snow, HAND drainage)
3. **Debris flow & optical fusion** (Sentinel-2 MNDWI & NDVI vegetation drop)
4. **OSM infrastructure intersection** (pre-event snapshot via ohsome before 2026-07-27)
5. **Cut-off road network analysis** (newly cut off vs rerouted vs never connected)
6. **Kuro Siwo U-Net benchmark & error analysis**
7. **Bonus: Downstream DEM flood-path tracing**

In [ ]:
import sys, os, json
sys.path.append('..')

from run_pipeline import run_pipeline
from src.model.unet_kuro_siwo import ModelBenchmark
from src.mapping.change_detection import FloodDebrisMapper
from src.bonus.flow_tracing import run_flood_path_tracing

bbox = [85.10, 27.85, 85.50, 28.15]
flood_date = '2026-08-26'
results = run_pipeline(bbox=bbox, event_date=flood_date, offline=True)
print('Certified Facts:', json.dumps(results['certified_facts'], indent=2))

In [ ]:
# Model Benchmark: Kuro Siwo U-Net vs Otsu Baselines
mapper = FloodDebrisMapper()
scene = mapper.generate_synthetic_scene(bbox)
bench = ModelBenchmark.run_benchmark(scene)

print('Segmentation Model Metrics:')
for model, metrics in bench['benchmark_results'].items():
    print(f"{model:40s}: IoU={metrics['iou']:.4f}, F1={metrics['f1_score']:.4f}")

In [ ]:
# Bonus Feature: DEM Flow Tracing
from src.data.acquisition import DataAcquisitionPipeline
osm = DataAcquisitionPipeline().fetch_or_synthesize_osm(bbox)
flow_res = run_flood_path_tracing(85.492, 27.955, bbox, osm)
print(f"Traced downstream flood wave: {flow_res['trace']['total_path_length_km']} km")
print('Impacted settlements along path:', [s['name'] for s in flow_res['impacts']['impacted_settlements']])